# 03. Підготовка даних

На цьому етапі створюємо й перевіряємо перетворення, не навчаючи класифікатор. Відкладена тестова вибірка не використовується. Демонстрація навчає preprocessing лише на навчальній частині одного фолду; для наступних експериментів pipeline потрібно навчати заново в кожному фолді.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import StratifiedKFold

project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent
sys.path.insert(0, str(project_root))
from src.data_split import load_dataset, split_dataset
from src.preprocessing import build_preprocessor, prepare_xy

data_path = project_root / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
dataset = load_dataset(data_path)
train, test = split_dataset(dataset)
del dataset, test
X, y = prepare_xy(train)
print(f"Навчальних записів: {len(X)}; початкових ознак: {X.shape[1]}")
print("Ціль: No → 0, Yes → 1")

Навчальних записів: 5634; початкових ознак: 19
Ціль: No → 0, Yes → 1


## Як працює pipeline

- `customerID` вилучається, а `Churn` відокремлюється як ціль.
- Порожні рядки `TotalCharges` перетворюються на пропуски; нечислові значення викликають помилку. Числові пропуски заповнюються медіаною навчального фолду, після чого ознаки масштабуються.
- `SeniorCitizen` обробляється як категорія; категоріальні пропуски заповнюються окремим значенням, категорії кодуються one-hot. Невідомі категорії на етапі transform не викликають помилки.
- Жодне перетворення не змінює сирий CSV; остаточний вибір ознак і навчання моделей залишаються на наступний етап.

In [2]:
folds = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fit_idx, validation_idx = next(folds.split(X, y))
X_fit, X_validation = X.iloc[fit_idx], X.iloc[validation_idx]
preprocessor = build_preprocessor()
fit_features = preprocessor.fit_transform(X_fit)
validation_features = preprocessor.transform(X_validation)
print(f"Навчальний фолд: {fit_features.shape}")
print(f"Валідаційний фолд: {validation_features.shape}")
print(f"Ознак після перетворення: {len(preprocessor.get_feature_names_out())}")
display(pd.DataFrame({"перші ознаки": preprocessor.get_feature_names_out()[:15]}))
imputer = preprocessor.named_steps["columns"].named_transformers_["numeric"].named_steps["imputer"]
print("Медіани навчального фолду (tenure, MonthlyCharges, TotalCharges):", imputer.statistics_)

Навчальний фолд: (4507, 46)
Валідаційний фолд: (1127, 46)
Ознак після перетворення: 46


,перші ознаки
0,numeric__tenure
1,numeric__MonthlyCharges
2,numeric__TotalCharges
3,categorical__gender_Female
4,categorical__gender_Male
5,categorical__SeniorCitizen_0
6,categorical__SeniorCitizen_1
7,categorical__Partner_No
8,categorical__Partner_Yes
9,categorical__Dependents_No


Медіани навчального фолду (tenure, MonthlyCharges, TotalCharges): [  29.    70.3 1387.4]


## Перевірка невідомої категорії та пропуску

Використовуємо копію одного запису з валідаційного фолду. Статистика preprocessing при цьому не перенавчається.

In [3]:
probe = X_validation.iloc[[0]].copy()
probe.loc[:, "Contract"] = "Previously unseen contract"
probe.loc[:, "TotalCharges"] = " "
result = preprocessor.transform(probe)
array = result.toarray() if hasattr(result, "toarray") else result
assert np.isfinite(array).all()
assert array.shape[1] == len(preprocessor.get_feature_names_out())
print("Перевірка пропуску та невідомої категорії пройдена.")

Перевірка пропуску та невідомої категорії пройдена.


## Далі

Для крос-валідації створюйте `Pipeline([('preprocess', build_preprocessor()), ('model', estimator)])` і передавайте необроблені ознаки `X`. Не виконуйте `fit_transform` на всьому `X` до крос-валідації: це призведе до витоку. Тестова вибірка залишається відкладеною; вона не є повністю сліпою через попередній огляд усього датасету.